# FASHN VTON v1.5 — Virtual Try-On on Kaggle

This notebook runs FASHN VTON v1.5 inference on Kaggle's free T4 GPU.

**Prerequisites:**
1. Add your Hugging Face token as a Kaggle secret named `HF_TOKEN`
   2. Enable GPU: Settings → Accelerator → T4 x2
   3. Upload this notebook to Kaggle and run all cells

---
⚠️ **Do not commit HF_TOKEN to GitHub.** Use Kaggle secrets only.


In [ ]:
# Install fashn-vton dependencies
!pip install -q torch torchvision safetensors huggingface_hub pillow numpy opencv-python tqdm einops onnxruntime matplotlib fashn-human-parser

# Clone fashn-vton repository
!git clone -q https://github.com/fashn-AI/fashn-vton-1.5.git
%cd fashn-vton-1.5

# Install package (no deps - already installed above)
!pip install -q -e . --no-deps


In [ ]:
# Authenticate with Hugging Face using Kaggle secret
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("✓ Hugging Face authenticated")
else:
    print("⚠️ HF_TOKEN not found. Weight download may be rate-limited.")


In [ ]:
# Download model weights (~2 GB)
!python scripts/download_weights.py --weights-dir ./weights

# Verify weights exist
!ls -lh weights/model.safetensors weights/dwpose/

In [ ]:
# Define our 12 test pairs from the VITON-HD dataset
# These match ai-research/dataset/test_pairs/test_pairs.json
test_pairs = [
    {"pair_id": "P01", "person": "00891_00.jpg", "garment": "01260_00.jpg"},
    {"pair_id": "P02", "person": "03615_00.jpg", "garment": "01430_00.jpg"},
    {"pair_id": "P03", "person": "07445_00.jpg", "garment": "02783_00.jpg"},
    {"pair_id": "P04", "person": "07573_00.jpg", "garment": "03751_00.jpg"},
    {"pair_id": "P05", "person": "08909_00.jpg", "garment": "06429_00.jpg"},
    {"pair_id": "P06", "person": "10549_00.jpg", "garment": "06802_00.jpg"},
    {"pair_id": "P07", "person": "00891_00.jpg", "garment": "07429_00.jpg"},
    {"pair_id": "P08", "person": "03615_00.jpg", "garment": "08348_00.jpg"},
    {"pair_id": "P09", "person": "07445_00.jpg", "garment": "09933_00.jpg"},
    {"pair_id": "P10", "person": "07573_00.jpg", "garment": "11028_00.jpg"},
    {"pair_id": "P11", "person": "08909_00.jpg", "garment": "11351_00.jpg"},
    {"pair_id": "P12", "person": "10549_00.jpg", "garment": "11791_00.jpg"},
]

print(f"Total test pairs: {len(test_pairs)}")


In [ ]:
# Run inference on all 12 test pairs
from fashn_vton import TryOnPipeline
from PIL import Image
import time, json, os

# Initialize pipeline
pipeline = TryOnPipeline(weights_dir="./weights", device="cuda")

# Create output directory
os.makedirs("outputs", exist_ok=True)

results = []

for pair in test_pairs:
    pair_id = pair["pair_id"]
    person_path = f"../ai-research/dataset/people/{pair['person']}"
    garment_path = f"../ai-research/dataset/clothing/{pair['garment']}"
    
    print(f"\n=== {pair_id}: {pair['person']} + {pair['garment']} ===")
    
    try:
        person_img = Image.open(person_path).convert("RGB")
        garment_img = Image.open(garment_path).convert("RGB")
        
        start = time.time()
        result = pipeline(
            person_image=person_img,
            garment_image=garment_img,
            category="tops",
            num_samples=1,
            num_timesteps=30,
            guidance_scale=1.5,
            seed=42,
            segmentation_free=True,
        )
        elapsed = time.time() - start
        
        # Save output
        output_path = f"outputs/{pair_id}_output.png"
        result.images[0].save(output_path)
        
        results.append({
            "pair_id": pair_id,
            "person": pair["person"],
            "garment": pair["garment"],
            "status": "success",
            "processing_time_seconds": round(elapsed, 2),
            "output_path": output_path,
        })
        print(f"✓ Completed in {elapsed:.1f}s → {output_path}")
        
    except Exception as e:
        results.append({
            "pair_id": pair_id,
            "person": pair["person"],
            "garment": pair["garment"],
            "status": "failed",
            "error": str(e),
        })
        print(f"✗ Failed: {e}")

# Save results summary
with open("outputs/results_summary.json", "w") as f:
    json.dump(results, f, indent=2)

successful = sum(1 for r in results if r["status"] == "success")
print(f"\n=== Summary: {successful}/{len(results)} pairs completed ===")


In [ ]:
# Verify outputs
!ls -lh outputs/*.png 2>/dev/null || echo "No output images yet"
!cat outputs/results_summary.json 2>/dev/null || echo "No results summary yet"
